# 1. Loading all the necessary libraries

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
from PIL import Image

# 2. Loading the dataset from Kaggle directly

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("drsaeedmohsen/alzheimer-dataset-four-classes-2025")

print("Path to dataset files:", path)

100%|██████████| 28.1M/28.1M [00:00<00:00, 243MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/drsaeedmohsen/alzheimer-dataset-four-classes-2025/versions/1


In [ ]:
path = "/root/.cache/kagglehub/datasets/drsaeedmohsen/alzheimer-dataset-four-classes-2025/versions/1"

train_dir = os.path.join(path, "---Dataset", "train")
test_dir  = os.path.join(path, "---Dataset", "test")

In [ ]:
def remove_corrupted_images(directory):
    for root, dirs, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                img = Image.open(file_path)
                img.verify()
            except:
                os.remove(file_path)

remove_corrupted_images(train_dir)
remove_corrupted_images(test_dir)

# 3. Transformations needed in preprocessing the dataset which also includes data augmentation

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [ ]:
train_dataset = ImageFolder(train_dir, transform=train_transforms)
test_dataset  = ImageFolder(test_dir, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 4. Loading RESNET50 Model

In [ ]:

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Unfreeze last layers
for param in model.parameters():
    param.requires_grad = False

for param in model.layer4.parameters():
    param.requires_grad = True

# Modify FC layer
model.fc = nn.Linear(model.fc.in_features, 4)
model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 202MB/s]


# 5. Training by experimenting with multiple optimizers and learning rates to get the best accuracy

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)


In [ ]:
optimizer1= torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
labels = np.array(train_dataset.targets)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

optimizer3 = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion1 = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
optimizer2= torch.optim.SGD(model.parameters(), lr=1e-4)

### 5. a) The below code represents the model being trained using cross entropy loss as the loss function and ADAM optimiser. The learning rate is kept to 0.0003

**RESULT : VALIDATION ACCURACY - 94.06%**

In [ ]:
epochs = 30

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0

    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

val_acc = correct / total

print(f"Epoch {epoch+1}")
print(f"Loss: {total_loss:.4f}")
print(f"Train Acc: {train_acc:.4f}")
print(f"Val Acc: {val_acc:.4f}")
print("-"*40)

Epoch 30
Loss: 29.7399
Train Acc: 0.9317
Val Acc: 0.8194
----------------------------------------


###5. b) The below code represents the model being trained using cross entropy loss as the loss function and ADAM optimiser. The learning rate is changed to 0.0001 for better accuracy.

**RESULT : VALIDATION ACCURACY - 96.25%**


In [ ]:
epochs = 30

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer1.zero_grad()
        loss.backward()
        optimizer1.step()

        total_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

val_acc = correct / total

print(f"Epoch {epoch+1}")
print(f"Loss: {total_loss:.4f}")
print(f"Train Acc: {train_acc:.4f}")
print(f"Val Acc: {val_acc:.4f}")
print("-"*40)

Epoch 30
Loss: 9.1051
Train Acc: 0.9826
Val Acc: 0.9296
----------------------------------------


###5. c) The below code represents the model being trained using cross entropy loss as the loss function and ADAM optimiser. Here we have used class weights of the pretrained model also.


**RESULT : VALIDATION ACCURACY - 95.15%**

In [ ]:
epochs = 30

train_acc_list = []
val_acc_list = []
loss_list = []

for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer3.zero_grad()
        outputs = model(images)
        loss = criterion1(outputs, labels)

        loss.backward()
        optimizer3.step()

        total_train_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)

    train_acc = correct_train / total_train
    train_acc_list.append(train_acc)
    loss_list.append(total_train_loss / len(train_loader))


In [ ]:
model.eval()
correct = 0
total = 0

val_acc_list = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

val_acc = correct / total
val_acc_list.append(val_acc)

print(f"Epoch {epoch+1}")
print(f"Loss: {total_loss:.4f}")
print(f"Train Acc: {train_acc:.4f}")
print(f"Val Acc: {val_acc:.4f}")
print("-"*40)

# 6. DATA VISUALISATION

### A) Predicted and true labels are displayed on top of the brain mri images

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random
import torch

# Class names
class_names = ['NonDemented', 'VeryMildDemented', 'MildDemented', 'ModerateDemented']

# Normalization parameters used during preprocessing
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

# Ensure the model is in evaluation mode
model.eval()

all_images = []
all_true_labels = []
all_pred_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        # Collect images (converted to numpy) and labels
        # Unnormalize and permute for display
        unnormalized_images = images.cpu().permute(0, 2, 3, 1).numpy() * std + mean
        all_images.extend(unnormalized_images)
        all_true_labels.extend(labels.cpu().numpy())
        all_pred_labels.extend(preds.cpu().numpy())

# Show multiple images
plt.figure(figsize=(12, 8))

# Select 6 random indices from the collected test data
# Make sure there are enough samples
num_samples_to_show = min(6, len(all_images))
random_indices = random.sample(range(len(all_images)), num_samples_to_show)

for i, idx_in_collected in enumerate(random_indices):
    img_display = all_images[idx_in_collected]
    true_label = all_true_labels[idx_in_collected]
    pred_label = all_pred_labels[idx_in_collected]

    # Clip values to [0, 1] in case unnormalization resulted in slight out-of-range values
    img_display = np.clip(img_display, 0, 1)

    # Handle grayscale images if they are still 3-channel after normalization
    # Matplotlib expects 2D for grayscale or 3D (H,W,C) for RGB
    if img_display.shape[-1] == 3 and (img_display[:,:,0] == img_display[:,:,1]).all() and (img_display[:,:,1] == img_display[:,:,2]).all():
        # If all channels are identical, assume it's a grayscale image that was duplicated to 3 channels
        img_display = img_display[:,:,0]

    # Color logic
    color = 'green' if true_label == pred_label else 'red'

    plt.subplot(2, 3, i + 1)
    plt.imshow(img_display)
    plt.title(f"Actual: {class_names[true_label]}\nPred: {class_names[pred_label]}", color=color)
    plt.axis('off')

plt.tight_layout()
plt.show()

### B) Confusion matrix : To see true and false positives and negatives
(visualise class imbalance as well)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

y_true = all_true_labels
y_pred_classes = all_pred_labels

cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

### C) Classification report

In [ ]:
from sklearn.metrics import classification_report
report = classification_report(y_true, y_pred_classes, output_dict=True)
print(report)

### D) Precision, F1 Score, Recall displayed per class

In [ ]:


import pandas as pd
df = pd.DataFrame(report).transpose()

df[['precision','recall','f1-score']].plot(kind='bar')
plt.title('Precision, Recall, F1-score per class')
plt.ylabel('Score')
plt.show()

### E) Training Loss visualisation graph

In [ ]:
plt.plot(loss_list, label='Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.show()

In [ ]:
import joblib

joblib.dump(model, 'model.pkl')

